# ml-pre-post-covid-analysis



## Startup cells

In [0]:
# Set environment variables for sagemaker_studio imports

import os
os.environ['DataZoneProjectId'] = 'bezvf4eioc4e93'
os.environ['DataZoneDomainId'] = 'dzd-6uj8apuu2tfsvb'
os.environ['DataZoneEnvironmentId'] = 'b1ikk5lqt0v50n'
os.environ['DataZoneDomainRegion'] = 'us-east-1'

# create both a function and variable for metadata access
_resource_metadata = None

def _get_resource_metadata():
    global _resource_metadata
    if _resource_metadata is None:
        _resource_metadata = {
            "AdditionalMetadata": {
                "DataZoneProjectId": "bezvf4eioc4e93",
                "DataZoneDomainId": "dzd-6uj8apuu2tfsvb",
                "DataZoneEnvironmentId": "b1ikk5lqt0v50n",
                "DataZoneDomainRegion": "us-east-1",
            }
        }
    return _resource_metadata
metadata = _get_resource_metadata()

In [0]:
"""
Logging Configuration

Purpose:
--------
This sets up the logging framework for code executed in the user namespace.
"""

from typing import Optional


def _set_logging(log_dir: str, log_file: str, log_name: Optional[str] = None):
    import os
    import logging
    from logging.handlers import RotatingFileHandler

    level = logging.INFO
    max_bytes = 5 * 1024 * 1024
    backup_count = 5

    # fallback to /tmp dir on access, helpful for local dev setup
    try:
        os.makedirs(log_dir, exist_ok=True)
    except Exception:
        log_dir = "/tmp/kernels/"

    os.makedirs(log_dir, exist_ok=True)
    log_path = os.path.join(log_dir, log_file)

    logger = logging.getLogger() if not log_name else logging.getLogger(log_name)
    logger.handlers = []
    logger.setLevel(level)

    formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")

    # Rotating file handler
    fh = RotatingFileHandler(filename=log_path, maxBytes=max_bytes, backupCount=backup_count, encoding="utf-8")
    fh.setFormatter(formatter)
    logger.addHandler(fh)

    logger.info(f"Logging initialized for {log_name}.")


_set_logging("/var/log/computeEnvironments/kernel/", "kernel.log")
_set_logging("/var/log/studio/data-notebook-kernel-server/", "metrics.log", "metrics")

In [0]:
import logging
from sagemaker_studio import ClientConfig, sqlutils, sparkutils, dataframeutils

logger = logging.getLogger(__name__)
logger.info("Initializing sparkutils")
spark = sparkutils.init()
logger.info("Finished initializing sparkutils")

In [0]:
def _reset_os_path():
    """
    Reset the process's working directory to handle mount timing issues.
    
    This function resolves a race condition where the Python process starts
    before the filesystem mount is complete, causing the process to reference
    old mount paths and inodes. By explicitly changing to the mounted directory
    (/home/sagemaker-user), we ensure the process uses the correct, up-to-date
    mount point.
    
    The function logs stat information (device ID and inode) before and after
    the directory change to verify that the working directory is properly
    updated to reference the new mount.
    
    Note:
        This is executed at module import time to ensure the fix is applied
        as early as possible in the kernel initialization process.
    """
    try:
        import os
        import logging

        logger = logging.getLogger(__name__)
        logger.info("---------Before------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)

        os.chdir("/home/sagemaker-user")

        logger.info("---------After------")
        logger.info("CWD: %s", os.getcwd())
        logger.info("stat('.'): %s %s", os.stat('.').st_dev, os.stat('.').st_ino)
        logger.info("stat('/home/sagemaker-user'): %s %s", os.stat('/home/sagemaker-user').st_dev, os.stat('/home/sagemaker-user').st_ino)
    except Exception as e:
        logger.exception(f"Failed to reset working directory: {e}")

_reset_os_path()

## Notebook

In [0]:
%pip install --upgrade databricks-sql-connector

  Installing build dependencies ... -

 \

 done


  Getting requirements to build wheel ... -

 done


...

   ━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/6 [oauthlib]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━ 4/6 [openpyxl]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╺━━━━━━ 5/6 [databricks-sql-connector]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6/6 [databricks-sql-connector]


Note: you may need to restart the kernel to use updated packages.


In [0]:
import boto3
import os
from botocore.exceptions import ClientError

def diagnosticar_conexion():
    try:
        # Este script se utilizó para testear el acceso a los secretos necesarios para conectar con databricks, ya que al inicio nos daba error
        client = boto3.client('secretsmanager', region_name='us-east-1') # Cambia tu región
        print("- Cliente de AWS creado.")
        
        client.describe_secret(SecretId='databricks/sql/credentials')
        print("- Acceso al secreto verificado.")
        
    except ClientError as e:
        print(f"Error de AWS: {e.response['Error']['Message']}")
    except Exception as e:
        print(f"Error inesperado: {str(e)}")

diagnosticar_conexion()

✓ Cliente de AWS creado.
✓ Acceso al secreto verificado.


In [0]:
import os
import logging
import json
import boto3
from botocore.exceptions import ClientError
from databricks import sql

# Logs
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# Este script se utilizo para testear la conexión efectiva con el SQL warehouse de databricks (nuestro gold layer)
class DatabricksConnector:
    
    def __init__(self, secret_name="databricks/sql/credentials", region_name="us-east-1"):
        self.secret_data = self._get_secret(secret_name, region_name)
        
        # Mapeo de credenciales desde el JSON que retorna AWS
        self.host = self.secret_data.get("DATABRICKS_HOST")
        self.http_path = self.secret_data.get("DATABRICKS_HTTP_PATH")
        self.access_token = self.secret_data.get("DATABRICKS_TOKEN")

    def _get_secret(self, secret_name, region_name):
        session = boto3.session.Session()
        client = session.client(service_name='secretsmanager', region_name=region_name)

        try:
            get_secret_value_response = client.get_secret_value(SecretId=secret_name)
        except ClientError as e:
            logger.error(f"Error al obtener secreto de AWS: {e}")
            raise e

        # Retorna el diccionario de credenciales
        return json.loads(get_secret_value_response['SecretString'])

    def execute_query(self, query):
        connection = None
        results = []
        
        try:
            logger.info("Iniciando conexión segura con Databricks...")
            connection = sql.connect(
                server_hostname=self.host,
                http_path=self.http_path,
                access_token=self.access_token
            )
            
            with connection.cursor() as cursor:
                cursor.execute(query)
                results = cursor.fetchall()
                logger.info(f"Consulta ejecutada. Filas: {len(results)}")
                
        except Exception as e:
            logger.error(f"Error crítico en la ejecución: {e}")
            raise
        finally:
            if connection:
                connection.close()
                logger.info("Conexión cerrada.")
        
        return results

# Ejemplo de uso:
if __name__ == "__main__":
    # Asegúrate de que el clúster tenga rol IAM asignado.
    connector = DatabricksConnector(secret_name="arn:aws:secretsmanager:us-east-1:769783602505:secret:databricks/sql/credentials-AhbpCO")
    try:
        data = connector.execute_query("SELECT count(*) FROM gold_ss2.fact_defunciones")
        print(f"Resultado: {data}")
    except Exception as e:
        print("No se pudo completar la operación.")

Resultado: [Row(count(1)=225106)]


In [0]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

# 1. Extracción desde Databricks
print("Extracción de datos de DW")

query = """
SELECT 
    g.pais,
    g.departamento,
    c.descripcion AS causa_mortalidad,
    d.sexo,
    d.grupo_etario_label AS grupo_etario,
    t.periodo_covid,
    f.total_defunciones
FROM gold_ss2.fact_defunciones f
INNER JOIN gold_ss2.dim_geografia g ON f.geografia_sk = g.geografia_sk
INNER JOIN gold_ss2.dim_causa c ON f.causa_sk = c.causa_sk
INNER JOIN gold_ss2.dim_tiempo t ON f.tiempo_sk = t.tiempo_sk
INNER JOIN gold_ss2.dim_demografia d ON f.demografia_sk = d.demografia_sk
WHERE f.total_defunciones > 0
"""

connector = DatabricksConnector(secret_name="arn:aws:secretsmanager:us-east-1:769783602505:secret:databricks/sql/credentials-AhbpCO")
data_raw = connector.execute_query(query)

# Convertir a DataFrame de Pandas
df = pd.DataFrame([row.asDict() for row in data_raw])
print(f"Datos obtenidos: {df.shape[0]} registros.")

print("Procesamiento de variables")
df['is_post_covid'] = np.where(df['periodo_covid'] == 'covid_y_post', 1, 0)
df['pais_causa'] = df['pais'] + "_" + df['causa_mortalidad']

columnas_categoricas = ['pais', 'departamento', 'causa_mortalidad', 'sexo', 'grupo_etario', 'pais_causa']
X = df[columnas_categoricas]
y = df['is_post_covid']
pesos = df['total_defunciones']  # El multiplicador clave

X_train, X_test, y_train, y_test, pesos_train, pesos_test = train_test_split(
    X, y, pesos, test_size=0.2, random_state=42, stratify=y
)

preprocesador = ColumnTransformer(
    transformers=[('cat', OneHotEncoder(handle_unknown='ignore'), columnas_categoricas)]
)

modelo_pipeline = Pipeline(steps=[
    ('preprocesamiento', preprocesador),
    ('clasificador', LogisticRegression(penalty='l1', C=0.1, solver='liblinear', random_state=42, max_iter=1000))
])

print("Entrenamiento de Modelo Logístico Ponderado (Lasso)\n")
modelo_pipeline.fit(X_train, y_train, clasificador__sample_weight=pesos_train)

# Datos de output del modelo
print("--- RENDIMIENTO DEL MODELO ---")
predicciones = modelo_pipeline.predict(X_test)
print(classification_report(y_test, predicciones, sample_weight=pesos_test))

print("\n--- CONCLUSIONES PARA POLÍTICAS PÚBLICAS ---")
nombres_features = modelo_pipeline.named_steps['preprocesamiento'].get_feature_names_out()
coeficientes = modelo_pipeline.named_steps['clasificador'].coef_[0]

importancias = pd.DataFrame({'Variable': nombres_features, 'Odds_Ratio': np.exp(coeficientes)})
importancias = importancias[importancias['Odds_Ratio'] != 1.0]  # Filtrar lo que Lasso anuló
top_cambios = importancias.sort_values(by='Odds_Ratio', ascending=False).head(15)

print("Estas características definen la era POST-COVID (Aumentaron su proporción de letalidad):")
print(top_cambios.to_string(index=False))

print("\nAlmacenamiento del modelo")
joblib.dump(modelo_pipeline, "modelo_logistico_ponderado.joblib")
print("Finalización")


Extracción de datos de DW


Datos obtenidos: 122423 registros.
Procesamiento de variables
Entrenamiento de Modelo Logístico Ponderado (Lasso)



/sagemaker_packages/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
/sagemaker_packages/.venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


KeyboardInterrupt: 

## Shutdown cells

In [0]:
"""
Stop spark session and associated Athena Spark session
"""

from IPython import get_ipython as _get_ipython
_get_ipython().user_ns["spark"].stop()